In [ ]:
# Import Necessary Packages/Modules
import time, numpy as np
import pandas as pd
from skeLCS import eLCS # Run "pip install scikit-eLCS" to install necessary module
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score

In [2]:
# Define the leakage columns again
LEAKAGE_COLS = ["is_high_risk","annual_premium","monthly_premium","annual_medical_cost",
                "claims_count","avg_claim_amount","total_claims_paid"]

# Read file into a table. Turns off default behavior that reads "none" as missing
df = pd.read_csv("medical_insurance.csv", keep_default_na=False, na_values=[""])

In [3]:
# Making rough thirds of risk score for stratify to work from
tmp_band = pd.qcut(df["risk_score"], 3, labels=False)
# Splitting 80% for training, 20% for testing and seeding every test for repeatable results
train, test = train_test_split(df, test_size=0.2, stratify=tmp_band, random_state=42)

In [4]:
# eLCS predicts classes, so we're cutting risk_score into 3 groups
q = train["risk_score"].quantile([1/3, 2/3]).values
edges = [-np.inf, q[0], q[1], np.inf] # The np.inf is just saying no upper or lower limits
y_train = pd.cut(train["risk_score"], edges, labels=False).values
y_test  = pd.cut(test["risk_score"],  edges, labels=False).values
print("class shares:", np.round(np.bincount(y_train) / len(y_train), 3)) # Imbalance check

class shares: [0.35  0.323 0.326]


In [5]:
# Quality checks
print("duplicates:", df.duplicated().sum(), "| nulls:", df.isna().sum().sum())

duplicates: 0 | nulls: 0


In [6]:
# Data-quality and outlier evidence
print("column types:", train.dtypes.astype(str).value_counts().to_dict())
for c in train.select_dtypes(exclude="number"):
    print(c, sorted(train[c].unique())) # looks for spelling or case variants
print("negative values anywhere:", bool((train.select_dtypes("number") < 0).any().any()))

infants = (train["age"] <= 1).values
low_bmi = (train["bmi"] < 13).values
print("infants:", infants.sum(), np.round(np.bincount(y_train[infants], minlength=3) / infants.sum(), 2))
print("BMI < 13:", low_bmi.sum(), np.round(np.bincount(y_train[low_bmi], minlength=3) / low_bmi.sum(), 2))

column types: {'int64': 31, 'float64': 13, 'str': 10}
sex ['Female', 'Male', 'Other']
region ['Central', 'East', 'North', 'South', 'West']
urban_rural ['Rural', 'Suburban', 'Urban']
education ['Bachelors', 'Doctorate', 'HS', 'Masters', 'No HS', 'Some College']
marital_status ['Divorced', 'Married', 'Single', 'Widowed']
employment_status ['Employed', 'Retired', 'Self-employed', 'Unemployed']
smoker ['Current', 'Former', 'Never']
alcohol_freq ['Daily', 'None', 'Occasional', 'Weekly']
plan_type ['EPO', 'HMO', 'POS', 'PPO']
network_tier ['Bronze', 'Gold', 'Platinum', 'Silver']
negative values anywhere: False
infants: 172 [0.92 0.07 0.01]
BMI < 13: 214 [0.37 0.32 0.31]


In [7]:
# Drop leaky and ID columns, and encode text as integer codes
def prepare(d):
    X = d.drop(columns=LEAKAGE_COLS + ["risk_score", "person_id"]).copy()
    for c in X.select_dtypes(exclude="number"):
        X[c] = X[c].astype("category").cat.codes
    return X

X_train, X_test = prepare(train), prepare(test)

In [8]:
# Feature selection on the training set only
tree = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_train, y_train) # A shallow tree finds which columns separate the bands
importance = pd.Series(tree.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importance.head(6).round(3))
selected = list(importance.index[:4])

chronic_count    0.405
age              0.366
smoker           0.161
bmi              0.069
sex              0.000
income           0.000
dtype: float64


In [9]:
# All Features vs top 4
def run(name, cols):
    t0 = time.time()
    m = eLCS(random_state=42).fit(X_train[cols].values.astype(float), y_train)
    s = balanced_accuracy_score(y_test, m.predict(X_test[cols].values.astype(float)))
    print(f"{name}: test balanced accuracy {s:.3f} | {time.time()-t0:.0f}s")

run("Before (45 features)", list(X_train.columns))
run("After (4 selected)", selected)

Before (45 features): test balanced accuracy 0.491 | 45s
After (4 selected): test balanced accuracy 0.773 | 37s
